# SKG MCP Client Smoke Test

This notebook tests the typed `SKGMCPClient` with either `stdio` (recommended) or `streamable-http`.


In [ ]:
import asyncio
import importlib
import json
import subprocess
import sys
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "skg_mcp").exists() and (PROJECT_ROOT.parent / "src" / "skg_mcp").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    from skg_mcp.client import SKGMCPClient
except ModuleNotFoundError as exc:
    if exc.name not in {"mcp", "skg_mcp"}:
        raise

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
        check=True,
    )
    importlib.invalidate_caches()
    from skg_mcp.client import SKGMCPClient
    
from skg_mcp.models import (
    ExpandContextArgs,
    ExpandNeighborsArgs,
    GetAttributionArgs,
    GetProvenanceArgs,
    LexicalSearchArgs,
    NodeKind,
    SearchNodeType,
)


In [2]:
TRANSPORT = "stdio"  # "stdio" or "streamable-http"
HTTP_ENDPOINT = "http://127.0.0.1:8000/mcp"
STDIO_ENV = {"SKG_BACKEND": "llm_mock"}
STDIO_ARGS = ["run", "src/server.py"]

STDIO_CWD = str(PROJECT_ROOT)

QUERY = "attention"
print(f"Using PROJECT_ROOT={PROJECT_ROOT}")


Using PROJECT_ROOT=/home/nipdep/Dev/skg-mcp


In [3]:
def get_client_cm():
    if TRANSPORT == "stdio":
        return SKGMCPClient.connect_stdio(
            env=STDIO_ENV,
            args=STDIO_ARGS,
            cwd=STDIO_CWD,
        )
    if TRANSPORT == "streamable-http":
        return SKGMCPClient.connect_streamable_http(HTTP_ENDPOINT)
    raise ValueError(f"Unsupported transport: {TRANSPORT}")

async def run_client_smoke(query: str) -> dict[str, Any]:
    client_cm = get_client_cm()

    async with client_cm as client:
        tools = await client.list_tools()
        tool_names = [tool.name for tool in tools]

        metadata = await client.server_metadata()
        catalog = await client.tool_catalog()

        search = await client.lexical_search(
            LexicalSearchArgs(
                query=query,
                node_types=[SearchNodeType.CONCEPT, SearchNodeType.STATEMENT],
                limit=4,
            )
        )

        target_id = search.concepts[0].id if search.concepts else "c-mock-1"
        context = await client.expand_context(
            ExpandContextArgs(node_id=target_id, node_kind=NodeKind.CONCEPT, paper_id="p-mock-1")
        )
        neighbors = await client.expand_neighbors(
            ExpandNeighborsArgs(node_id=target_id, node_kind=NodeKind.CONCEPT, hop_count=2, limit=5)
        )
        attribution = await client.get_attribution(
            GetAttributionArgs(node_ids=[target_id], node_kind=NodeKind.CONCEPT)
        )
        provenance = await client.get_provenance(
            GetProvenanceArgs(node_ids=[target_id], node_kind=NodeKind.CONCEPT)
        )

        return {
            "transport": TRANSPORT,
            "tool_count": len(tool_names),
            "tools": tool_names,
            "server_metadata_name": metadata.get("name"),
            "catalog_size": len(catalog.get("tools", [])),
            "search": {
                "concept_count": len(search.concepts),
                "statement_count": len(search.statements),
            },
            "expand_context_has_node": context.node.id == target_id,
            "expand_neighbors_count": len(neighbors.neighbors),
            "attribution_count": len(attribution.attributions),
            "provenance_count": len(provenance.provenance),
        }


In [4]:
result = await run_client_smoke(query=QUERY)
print(json.dumps(result, indent=2))


{
  "transport": "stdio",
  "tool_count": 9,
  "tools": [
    "filter_papers",
    "lexical_search",
    "semantic_search",
    "semantic_constraint_search",
    "resolve_concept_reference",
    "expand_context",
    "expand_neighbors",
    "get_attribution",
    "get_provenance"
  ],
  "server_metadata_name": "Scholarly Knowledge Graph MCP",
  "catalog_size": 9,
  "search": {
    "concept_count": 2,
    "statement_count": 2
  },
  "expand_context_has_node": true,
  "expand_neighbors_count": 3,
  "attribution_count": 1,
  "provenance_count": 1
}


In [5]:

assert result["tool_count"] > 0
assert result["search"]["concept_count"] + result["search"]["statement_count"] > 0
print("Client smoke test passed")


Client smoke test passed


## Tool Contract And Direct Call

This section shows how to inspect MCP tool input/output schemas and then call a tool with raw MCP arguments.


In [6]:
TOOL_CONTRACT_TARGET = ["lexical_search"]

async with get_client_cm() as client:
    tools = await client.list_tools()
    by_name = {tool.name: tool for tool in tools}
    selected_tool_name = next((name for name in TOOL_CONTRACT_TARGET if name in by_name), None)
    if selected_tool_name is None:
        available = sorted(by_name.keys())
        raise RuntimeError(f"No target tool found. Available tools: {available}")

    selected_tool = by_name[selected_tool_name]
    tool_contract = {
        "name": selected_tool.name,
        "description": selected_tool.description,
        "input_schema": selected_tool.inputSchema,
        "output_schema": selected_tool.outputSchema,
    }

print(f"Selected tool: {selected_tool_name}")
print(json.dumps(tool_contract, indent=2)[:4000])


Selected tool: lexical_search
{
  "name": "lexical_search",
  "description": "Unified lexical search across concept and/or statement nodes using node_types.",
  "input_schema": {
    "$defs": {
      "ConceptFilter": {
        "properties": {
          "paper_ids": {
            "anyOf": [
              {
                "items": {
                  "type": "string"
                },
                "type": "array"
              },
              {
                "type": "null"
              }
            ],
            "default": null,
            "title": "Paper Ids"
          },
          "concept_types": {
            "anyOf": [
              {
                "items": {
                  "type": "string"
                },
                "type": "array"
              },
              {
                "type": "null"
              }
            ],
            "default": null,
            "title": "Concept Types"
          },
          "canonical_only": {
            "default": fa

In [7]:
tool_args = {
    "args": {
        "query": QUERY,
        "node_types": ["concept", "statement"],
        "limit": 3,
    }
}

async with get_client_cm() as client:
    call_result = await client.call_tool(selected_tool_name, tool_args)
    output_payload = call_result.structuredContent
    if output_payload is None:
        output_payload = {}

print(f"Tool call isError={call_result.isError}")
print("Input payload:")
print(json.dumps(tool_args, indent=2))
print("Output payload (structuredContent):")
print(json.dumps(output_payload, indent=2)[:4000])

print(
    "Output summary:",
    {
        "concepts": len(output_payload.get("concepts", [])),
        "statements": len(output_payload.get("statements", [])),
    },
)


Tool call isError=False
Input payload:
{
  "args": {
    "query": "attention",
    "node_types": [
      "concept",
      "statement"
    ],
    "limit": 3
  }
}
Output payload (structuredContent):
{
  "concepts": [
    {
      "id": "c-lex-1",
      "label": "attention",
      "aliases": null,
      "concept_type": "mock_concept",
      "is_canonical": true,
      "paper_id": null,
      "score": 0.99,
      "summary": "Synthetic concept generated for 'attention'.",
      "provenance": null
    }
  ],
  "statements": [
    {
      "id": "s-lex-1",
      "text": "Lexical match for 'attention'.",
      "statement_type": "mock_statement",
      "rhetorical_role": "method",
      "paper_id": null,
      "score": 0.875,
      "provenance": null
    },
    {
      "id": "s-lex-2",
      "text": "'attention' appears in method section.",
      "statement_type": "mock_statement",
      "rhetorical_role": "method",
      "paper_id": null,
      "score": 0.894,
      "provenance": null
    }
  ]